# 3.7 — Polynomial & basis-function regression

Basis functions keep the model linear in coefficients while letting the curve bend in input space.

Polynomial & basis-function regression turns empirical risk minimization into a practical curve-fitting workflow. The hypothesis is still linear in coefficients, but the features can encode powers, interactions, and other bases, so validation rather than training loss decides whether the extra bend helps.

Save a copy to Drive to edit.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler

np.random.seed(7)
random.seed(7)

from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

def reg_ladder():
    """D1..D5 regression ladder of rising complexity. Returns [(name, X, y), ...]."""
    rungs = []

    x1 = np.array([[0.0], [1.0], [2.0], [3.0]])
    y1 = np.array([1.0, 3.0, 5.0, 7.0])
    rungs.append(("D1 hand line y=2x+1", x1, y1))

    rng = np.random.default_rng(1)
    x2 = np.linspace(-3, 3, 120).reshape(-1, 1)
    y2 = (2.0 * x2[:, 0] + 1.0) + rng.normal(0, 0.5, size=120)
    rungs.append(("D2 linear + noise", x2, y2))

    x3 = np.linspace(-3, 3, 160).reshape(-1, 1)
    y3 = np.sin(1.5 * x3[:, 0]) + rng.normal(0, 0.2, size=160)
    rungs.append(("D3 sine (non-linear)", x3, y3))

    dia = load_diabetes()
    rungs.append(("D4 Diabetes (real, 10-D)", dia.data, dia.target))

    x5, y5 = make_regression(n_samples=300, n_features=20, n_informative=8, noise=25.0, random_state=5)
    rungs.append(("D5 high-dim + noise (20-D)", x5, y5))

    return rungs

def reg_rmse(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out RMSE."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return float(np.sqrt(mean_squared_error(y_te, preds)))

def linear_baseline(x_tr, y_tr, x_te):
    clf = LinearRegression()
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)

## The concept, built once on D1

The lesson formula is

$$\hat y=\sum_{j=0}^{p}\beta_j\phi_j(x)$$

First we reproduce the lesson arithmetic exactly: the three cited losses are 0.257, 0.109, and 0.488.

In [ ]:
lesson_losses = np.array([0.257, 0.109, 0.488])
lesson_total = round(float(lesson_losses.sum()), 3)
lesson_risk = round(lesson_total / 3.0, 3)
lesson_cost = 0.06
lesson_score = round(lesson_risk + lesson_cost, 3)
lesson_alternative = 0.389
lesson_gap = round(lesson_alternative - lesson_score, 3)
lesson_stabilized = round(0.8 * lesson_score, 3)

assert lesson_total == 0.854
assert lesson_risk == 0.285
assert lesson_score == 0.345
assert lesson_gap == 0.044
assert lesson_stabilized == 0.276

print("loss total", lesson_total)
print("empirical risk", lesson_risk)
print("cost-adjusted score", lesson_score)
print("validation gap", lesson_gap)
print("stabilized score", lesson_stabilized)

Now we package the real estimator as `polynomial_basis_function_regression_method()`. The method is reusable: it accepts training data and returns predictions for new rows, so the exact same call can run from D1 through D5.

In [ ]:
def polynomial_basis_function_regression_method(degree=3):
    model = make_pipeline(
        StandardScaler(),
        PolynomialFeatures(degree=degree, include_bias=False),
        LinearRegression(),
    )

    def build_and_predict(x_tr, y_tr, x_te):
        model.fit(x_tr, y_tr)
        return model.predict(x_te)

    return build_and_predict

## The dataset ladder

All six notebooks in this batch use `reg_ladder()`: D1 is inspectable, D2 is clean linear signal, D3 is nonlinear sine signal, D4 is sklearn's real diabetes regression dataset, and D5 is a real high-dimensional diabetes interaction design built without downloads. We report MSE as the lesson-plan metric, with `reg_rmse()` as the shared helper check and R² as a secondary annotation.

In [ ]:
rungs = reg_ladder()
diabetes = load_diabetes()
interaction_builder = PolynomialFeatures(degree=2, include_bias=False)
real_d5_X = interaction_builder.fit_transform(diabetes.data)
real_d5_y = diabetes.target
rungs[-1] = ("D5 Diabetes interactions (real, 65-D)", real_d5_X, real_d5_y)

for rung_index, (name, X, y) in enumerate(rungs, start=1):
    print(f"{rung_index}. {name}")
    print("  X shape", X.shape)
    print("  y size", y.shape[0])
    print("  X sample", np.round(X[:3], 3).tolist())
    print("  y sample", np.round(y[:3], 3).tolist())

## Run the same method across D1–D5

We compare degree-three basis features vs the no-basis linear baseline. The no-skill baseline is `linear_baseline`; the lesson method is `polynomial_basis_function_regression_method`. `reg_rmse()` is called on every rung to keep this regression batch tied to the shared helper.

In [ ]:
method = polynomial_basis_function_regression_method(degree=2)
baseline = linear_baseline
rows = []
prediction_store = []

for rung_index, (name, X, y) in enumerate(rungs, start=1):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
    baseline_pred = baseline(x_tr, y_tr, x_te)
    method_pred = method(x_tr, y_tr, x_te)
    baseline_mse = float(mean_squared_error(y_te, baseline_pred))
    method_mse = float(mean_squared_error(y_te, method_pred))
    baseline_rmse = reg_rmse(baseline, X, y)
    method_rmse = reg_rmse(method, X, y)
    assert abs(method_rmse - math.sqrt(method_mse)) < 1e-8
    method_r2 = float(r2_score(y_te, method_pred))
    rows.append((rung_index, name, baseline_rmse, method_rmse, method_mse, method_r2))
    prediction_store.append((name, X, y, x_te, y_te, method_pred))

print("rung | dataset | linear RMSE | method RMSE | method MSE | method R^2")
for rung_index, name, baseline_rmse, method_rmse, method_mse, method_r2 in rows:
    print(f"D{rung_index} | {name} | {baseline_rmse:.3f} | {method_rmse:.3f} | {method_mse:.3f} | {method_r2:.3f}")

## Results visualization

The closing figure has two parts: small multiples show the fitted output artifact on each rung, and the summary curve tracks MSE from D1 to D5.

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for col, (name, X, y, x_te, y_te, method_pred) in enumerate(prediction_store):
    axis = axes[0, col]
    order = np.argsort(x_te[:, 0])
    axis.scatter(x_te[:, 0], y_te, s=18, alpha=0.65, label="truth")
    axis.scatter(x_te[:, 0], method_pred, s=18, alpha=0.65, label="prediction")
    axis.set_title(f"D{col + 1} degree=3 polynomial/basis curve", fontsize=9)
    axis.set_xlabel("first feature")
    if col == 0:
        axis.set_ylabel("target")
        axis.legend(fontsize=8)

mse_values = [row[4] for row in rows]
axes[1, 0].plot(range(1, 6), mse_values, marker="o")
axes[1, 0].set_xticks(range(1, 6))
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("MSE")
axes[1, 0].set_title("MSE vs ladder complexity")

for empty_axis in axes[1, 1:]:
    empty_axis.axis("off")

plt.tight_layout()
plt.show()

## Pitfall on D5: optimizing the raw term and forgetting the cost

The lesson warns that raw empirical risk is not the whole selection score. On D5 we reproduce that mistake, then add the lesson cost/scale/gap check before choosing the winner.

In [ ]:
d5_name, d5_X, d5_y = rungs[-1]
x_tr, x_te, y_tr, y_te = train_test_split(d5_X, d5_y, test_size=0.4, random_state=0)
method_pred = method(x_tr, y_tr, x_te)
baseline_pred = baseline(x_tr, y_tr, x_te)
method_raw = math.sqrt(float(mean_squared_error(y_te, method_pred)))
baseline_raw = math.sqrt(float(mean_squared_error(y_te, baseline_pred)))
wrong_winner = "method" if method_raw < baseline_raw else "linear baseline"
scale = max(method_raw, baseline_raw, 1.0)
method_score = method_raw / scale + lesson_cost
baseline_score = baseline_raw / scale
fixed_winner = "method" if method_score < baseline_score else "linear baseline"
observed_gap = abs(method_score - baseline_score)

print("D5", d5_name)
print("wrong raw-only winner", wrong_winner)
print("method raw RMSE", round(method_raw, 3))
print("baseline raw RMSE", round(baseline_raw, 3))
print("lesson cost", lesson_cost)
print("cost-adjusted method score", round(method_score, 3))
print("cost-adjusted baseline score", round(baseline_score, 3))
print("fixed winner", fixed_winner)
print("observed adjusted gap", round(observed_gap, 3))
print("lesson minimum meaningful gap", lesson_gap)

if observed_gap < lesson_gap:
    print("decision: gap is too small; prefer the simpler setting or collect more validation data")
else:
    print("decision: adjusted gap clears the lesson check")

## Evaluate it + Practice

- Metric: MSE is the lesson-plan metric; `reg_rmse()` supplies the shared RMSE helper check and R² is secondary.
- No-skill baseline: compare against unregularized `linear_baseline` on every rung.
- Cheap sanity check: D1 should be explainable from the printed predictions; if it behaves oddly, inspect preprocessing, extrapolation, and the intercept.
- Ablation: turn off the key idea (degree-three basis features vs the no-basis linear baseline) and confirm the score or stability worsens on at least one harder rung.
- Failure signals: a tiny validation gap, scale-mismatched scores, or a D5 winner that changes after adding the lesson cost.

Practice 1: change one hyperparameter and rerun the ladder table. Which rung moves the most?

Practice 2: replace RMSE with MAE for the table. Does the D5 winner change?

Practice 3: add a short note explaining whether the lesson gap is large enough for deployment.